# MFCC-10 CNN UUV Detection

Trains temporal Conv1D models with grouped 4-fold cross-validation on MFCC-10 data.

In [ ]:
from pathlib import Path
import os
import sys
from urllib.request import urlretrieve

UTILS_GITHUB_RAW_BASE_URL = "https://raw.githubusercontent.com/Nabuhodonozzor/uuv-detection/main"  # Set to the repository raw-file URL when utility files are not synced to Colab.


def find_or_fetch_utility(relative_path: str) -> Path:
    roots = [Path.cwd(), Path.cwd().parent, Path("/content"), Path("/content/drive/MyDrive/STUDA/src")]
    for root in roots:
        candidate = root / relative_path
        if candidate.is_file():
            return candidate
    if UTILS_GITHUB_RAW_BASE_URL:
        destination = Path("/content") / relative_path
        destination.parent.mkdir(parents=True, exist_ok=True)
        urlretrieve(f"{UTILS_GITHUB_RAW_BASE_URL.rstrip('/')}/{relative_path}", destination)
        return destination
    raise FileNotFoundError(
        f"Missing {relative_path}. Sync it with the VS Code Colab extension, upload it to /content, "
        "mount Drive at /content/drive/MyDrive/STUDA/src, or set UTILS_GITHUB_RAW_BASE_URL."
    )


common_utils_path = find_or_fetch_utility("utils/common_utils.py")
mfcc_cnn_utils_path = find_or_fetch_utility("CNN/mfcc_cnn_utils.py")
for utility_dir in (common_utils_path.parent, mfcc_cnn_utils_path.parent):
    if str(utility_dir) not in sys.path:
        sys.path.insert(0, str(utility_dir))

from common_utils import (
    configure_kaggle_access, cross_validate_keras_models_for_variants,
    evaluate_models_for_variants, extract_zip, plot_training_histories,
    prepare_mfcc_dataset_variants, save_keras_artifacts,
    summarize_cross_validation, train_final_keras_models_for_variants,
    zip_artifacts,
)
from mfcc_cnn_utils import build_mfcc_cnn, get_mfcc_cnn_callbacks


In [ ]:
from google.colab import files
from IPython.display import display

DATASET_KEY = "mfcc10"
DATASET_LABEL = "MFCC-10"
DATASET_SLUG = "pawedyrda/mfcc10"
ARCHIVE_PATH = Path("/content/mfcc10.zip")
EPOCHS = 50
BATCH_SIZE = 64



In [ ]:
configure_kaggle_access("Kaggle")
!kaggle datasets download -d {DATASET_SLUG} -p /content --force
extract_zip(ARCHIVE_PATH, "/content")
DATA_PATH = Path("/content") / f"{DATASET_KEY}.npz"
if not DATA_PATH.is_file():
    raise FileNotFoundError(f"Expected feature archive: {DATA_PATH}")
print(f"Using feature archive: {DATA_PATH}")


In [ ]:
cv_dataset = prepare_mfcc_dataset_variants(DATA_PATH)
final_variants = cv_dataset.final_variants()
print(f"Split ID: {cv_dataset.split_id}")
print(f"CV folds: {cv_dataset.n_splits}")
print(f"MFCC input shape: {cv_dataset.normal.cv_data.shape[1:]}")


In [ ]:
cv_multilabel_results, multilabel_best_epochs, multilabel_cv_histories = cross_validate_keras_models_for_variants(
    build_mfcc_cnn, cv_dataset, "multilabel", EPOCHS, BATCH_SIZE,
    get_mfcc_cnn_callbacks, DATASET_LABEL,
)
multilabel_models, multilabel_histories = train_final_keras_models_for_variants(
    build_mfcc_cnn, cv_dataset, "multilabel", multilabel_best_epochs, BATCH_SIZE,
)
display(summarize_cross_validation(cv_multilabel_results))


In [ ]:
cv_binary_results, binary_best_epochs, binary_cv_histories = cross_validate_keras_models_for_variants(
    build_mfcc_cnn, cv_dataset, "binary", EPOCHS, BATCH_SIZE,
    get_mfcc_cnn_callbacks, DATASET_LABEL,
)
binary_models, binary_histories = train_final_keras_models_for_variants(
    build_mfcc_cnn, cv_dataset, "binary", binary_best_epochs, BATCH_SIZE,
)
display(summarize_cross_validation(cv_binary_results))


In [ ]:
multilabel_results = evaluate_models_for_variants(multilabel_models, final_variants, "multilabel", DATASET_LABEL)
binary_results = evaluate_models_for_variants(binary_models, final_variants, "binary", DATASET_LABEL)
comparison_results = pd.concat([
    multilabel_results.assign(task="multilabel"),
    binary_results.assign(task="binary"),
], ignore_index=True)
display(comparison_results[["task", "Model", "precision", "recall", "f1-score", "support"]])

plot_training_histories(multilabel_histories, f"Multilabel MFCC-CNN final training - {DATASET_LABEL}")
plot_training_histories(binary_histories, f"Binary MFCC-CNN final training - {DATASET_LABEL}")

split_metadata = {
    "split_id": cv_dataset.split_id,
    "n_splits": cv_dataset.n_splits,
    "n_mfcc": cv_dataset.n_mfcc,
}
save_dir = save_keras_artifacts(
    f"/content/saved_artifacts/cnn_{DATASET_KEY}",
    DATASET_KEY,
    multilabel_models,
    binary_models,
    multilabel_histories,
    binary_histories,
    multilabel_results,
    binary_results,
    cv_multilabel_results=cv_multilabel_results,
    cv_binary_results=cv_binary_results,
    cv_histories={**multilabel_cv_histories, **binary_cv_histories},
    split_metadata=split_metadata,
)
comparison_results.to_csv(save_dir / f"cnn_comparison_{DATASET_KEY}.csv", index=False)
archive_path = zip_artifacts(save_dir, f"/content/cnn_models_and_results_{DATASET_KEY}.zip")
files.download(str(archive_path))
